# ReproPilot Interactive Repository Assessor
Paste a public GitHub URL, choose whether HPC checks apply, and click **Assess Repository**.


In [1]:
from pathlib import Path
from urllib.parse import urlparse
import json
import shutil
import subprocess
import sys
import tempfile
import time

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Add the repository root to Python's import path.
CURRENT_DIR = Path.cwd().resolve()

if (CURRENT_DIR / "checker").is_dir():
    REPO_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "checker").is_dir():
    REPO_ROOT = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Could not locate the ReproPilot repository root containing checker/."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from checker.reproducibility_checker import assess_repository
from checker.quality_assessor import assess_repository_quality

try:
    from checker.ai_priority_ranker import ai_priority_labels
    AI_AVAILABLE = True
except Exception:
    AI_AVAILABLE = False

print(f"ReproPilot root: {REPO_ROOT}")
print(f"Grounded AI available: {AI_AVAILABLE}")

ReproPilot root: /Users/suzananwar/Downloads/ai-assisted-reproducibility-bssw
Grounded AI available: True


## Reproducibility Checklist & Scoring Rubric


In [2]:
RUBRIC = [
    {"Category": "Documentation", "Item": "README with run instructions", "Points": 15},
    {"Category": "Dependencies", "Item": "requirements.txt", "Points": 15},
    {"Category": "Environment", "Item": "environment.yml", "Points": 10},
    {"Category": "HPC Environment", "Item": "spack.yaml", "Points": 10},
    {"Category": "Testing", "Item": "tests/ folder", "Points": 15},
    {"Category": "Containerization", "Item": "Dockerfile or Apptainer file", "Points": 15},
    {"Category": "Experiment Tracking", "Item": "MLflow logs or tracking file", "Points": 10},
    {"Category": "Open Science", "Item": "LICENSE file", "Points": 10},
]
rubric_df = pd.DataFrame(RUBRIC)
rubric_df.loc[len(rubric_df)] = ["Total", "", 100]
display(rubric_df)

,Category,Item,Points
0,Documentation,README with run instructions,15
1,Dependencies,requirements.txt,15
2,Environment,environment.yml,10
3,HPC Environment,spack.yaml,10
4,Testing,tests/ folder,15
5,Containerization,Dockerfile or Apptainer file,15
6,Experiment Tracking,MLflow logs or tracking file,10
7,Open Science,LICENSE file,10
8,Total,,100


In [3]:
def validate_github_url(url):
    parsed = urlparse(url.strip())
    if parsed.scheme != "https" or parsed.hostname != "github.com":
        raise ValueError("Use a public HTTPS GitHub URL.")
    parts = [p for p in parsed.path.strip("/").split("/") if p]
    if len(parts) < 2:
        raise ValueError("The URL must include owner and repository.")
    owner, repo = parts[0], parts[1].removesuffix(".git")
    return f"https://github.com/{owner}/{repo}.git", repo

def clone_repository(url):
    canonical, repo_name = validate_github_url(url)
    temp_root = Path(tempfile.mkdtemp(prefix="repropilot-notebook-"))
    destination = temp_root / repo_name
    subprocess.run(
        ["git", "clone", "--depth", "1", "--filter=blob:none", canonical, str(destination)],
        check=True, capture_output=True, text=True, timeout=180
    )
    return temp_root, destination, repo_name

def presence_table(presence):
    return pd.DataFrame([{
        "Category": x.get("label"),
        "Status": x.get("status"),
        "Score": f"{x.get('earned',0)}/{x.get('possible',0)}",
        "Evidence": ", ".join(x.get("found_paths", [])) or "None",
        "Recommendation": x.get("recommendation", "")
    } for x in presence.get("findings", [])])

def quality_table(quality):
    return pd.DataFrame([{
        "Category": x.get("label"),
        "Status": x.get("status"),
        "Score": "N/A" if not x.get("applicable", True) else f"{x.get('earned',0)}/{x.get('possible',0)}",
        "Percent": x.get("percent"),
        "Evidence": ", ".join(x.get("evidence", [])) or "None",
        "Recommendation": x.get("recommendation", "")
    } for x in quality.get("quality_findings", [])])

In [4]:
url_input = widgets.Text(
    value="https://github.com/szuananwar/ai-assisted-reproducibility-bssw",
    description="GitHub URL:",
    layout=widgets.Layout(width="850px"),
    style={"description_width": "100px"},
)
hpc_checkbox = widgets.Checkbox(value=True, description="Apply HPC-specific checks", indent=False)
ai_checkbox = widgets.Checkbox(
    value=False,
    description="Run grounded local AI",
    indent=False,
    disabled=not AI_AVAILABLE,
)
assess_button = widgets.Button(
    description="Assess Repository",
    button_style="primary",
    icon="check",
    layout=widgets.Layout(width="220px", height="42px"),
)
status_output = widgets.Output()
results_output = widgets.Output()

display(widgets.VBox([
    url_input,
    widgets.HBox([hpc_checkbox, ai_checkbox]),
    assess_button,
    status_output,
    results_output,
]))

In [5]:
def run_assessment(_):
    temp_root = None
    started = time.perf_counter()
    try:
        with status_output:
            clear_output()
            display(HTML("<b>Cloning and assessing repository...</b>"))

        temp_root, repo_path, repo_name = clone_repository(url_input.value)
        presence = assess_repository(repo_path)
        quality = assess_repository_quality(repo_path, hpc_applicable=hpc_checkbox.value)
        ai_result = ai_priority_labels(quality) if ai_checkbox.value else None

        elapsed = time.perf_counter() - started

        with status_output:
            clear_output()
            display(HTML(f"<b style='color:#047857'>Completed in {elapsed:.1f} seconds.</b>"))

        with results_output:
            clear_output()
            display(HTML(f"<h2>{repo_name}</h2>"))
            display(HTML(
                f"<h3>Presence Score: {presence.get('percent',0):.1f}% &nbsp;&nbsp; "
                f"Quality Score: {quality.get('quality_percent',0):.1f}%</h3>"
            ))
            display(HTML("<h3>Reproducibility Checklist Results</h3>"))
            display(presence_table(presence))
            display(HTML("<h3>Artifact Quality Results</h3>"))
            display(quality_table(quality))
            display(HTML("<h3>Top Deterministic Priorities</h3>"))
            display(pd.DataFrame(quality.get("priority_actions", [])))
            if ai_result is not None:
                display(HTML("<h3>Grounded AI Priorities</h3>"))
                display(HTML(f"<pre>{json.dumps(ai_result, indent=2)}</pre>"))

            report = {
                "repository_url": url_input.value,
                "presence": presence,
                "quality": quality,
                "ai": ai_result,
            }
            report_path = REPO_ROOT /"notebooks"/"latest_repropilot_assessment.json"
            report_path.parent.mkdir(parents=True, exist_ok=True)
            report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
            display(HTML(f"<p>JSON report saved to <code>{report_path}</code></p>"))

    except Exception as exc:
        with status_output:
            clear_output()
            display(HTML(f"<b style='color:#b91c1c'>Assessment failed: {type(exc).__name__}: {exc}</b>"))
    finally:
        if temp_root:
            shutil.rmtree(temp_root, ignore_errors=True)

assess_button.on_click(run_assessment)